# Evaluating Corbin Carroll’s Career Trajectory: Multi-Year WAR Forecasting and Retention via Conformal Prediction

## Purpose
This script executes a multi-step predictive framework to forecast Corbin Carroll’s future multi-year career Wins Above Replacement (WAR) and active retention status using historical Major League Baseball (MLB) player data. It implements **Conformal Prediction** to construct rigorous 90% prediction intervals and valid uncertainty sets.

## Experimental Setting
* **Dataset:** Historical MLB player performance and seasonal tracking metrics.
* **Target Features:** `Age`, `year` (season duration index), current `WAR`, and lagged metrics (`WAR-1`, `WAR-2`).
* **Target Horizons ($h$):** Multi-step forecasting horizons from $h = 1$ up to $h = 5$ years ahead.
* **Target Coverage:** $90\%$ ($\alpha = 0.10$).
* **Evaluation Framework:** Player-disjoint 80/10/10 train/calibration/test split evaluated over categories.

## Methods
1. **Data Cleaning & Feature Engineering:** Filters out incomplete historical logs, standardizes columns, handles player career continuity, and constructs lagged multi-year features ($WAR-1$, $WAR-2$) alongside active/retirement target labels.
2. **Model Training (Case 1 & Case 2):** 
   - *Case 1 (WAR Regression):* Predicts future cumulative seasonal WAR using Linear Regression, Neural Networks (MLP), and XGBoost Regressor.
   - *Case 2 (Active Classification):* Estimates player retention probabilities using Logistic Regression, MLP Classifier, and XGBoost Classifier.
3. **Conformal Uncertainty Quantification:** We use split conformal prediction and locally adaptively conformal prediction to computes nonconformity scores on the calibration set and determines validity quantiles ($q$) to build distribution-free prediction intervals for WAR and prediction sets for active status.
4. **Recursive Career Simulation (Algorithm 3):** Simulates Corbin Carroll's long-term career trajectory year-by-year from his baseline rookie seasons, dynamically updating feature vectors and evaluating stopping conditions when the active retention model predicts retirement.

## Output
* Generates console validation logs containing empirical coverage, test row counts, and conformal quantiles ($q$) for all regression and classification models across horizons $h = 1$ to $5$.
* Produces recursive longitudinal projection traces and visualizes career WAR trajectories with 90% conformal confidence intervals and estimated retirement markers.
* The following two blocks are the case for Split Conformal Prediction and Locally Adaptively Conformal Prediction.

## Split Conformal Prediction

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPRegressor, MLPClassifier
from xgboost import XGBRegressor, XGBClassifier

# --- Helper Functions ---
def df_to_md(df, index=False):
    """Converts DataFrame to GitHub Markdown table without external dependencies."""
    if index:
        df = df.reset_index()
    headers = [str(col) for col in df.columns]
    rows = []
    rows.append("| " + " | ".join(headers) + " |")
    rows.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in df.iterrows():
        str_row = [str(val) for val in row.values]
        rows.append("| " + " | ".join(str_row) + " |")
    return "\n".join(rows)

def get_conformal_quantile(scores, alpha=0.10):
    """Calculates nonconformity score quantile for 1-alpha target coverage."""
    n = len(scores)
    if n == 0:
        return 0.0
    q_val = np.ceil((n + 1) * (1 - alpha)) / n
    q_val = min(1.0, max(0.0, q_val))
    return np.quantile(scores, q_val, method='higher') if hasattr(np, 'quantile') else np.percentile(scores, q_val * 100)


def main():
    # --- Step 1: Load Data & Preprocess Features ---
    excel_filename = 'file name'
    if not os.path.exists(excel_filename):
        excel_filename = os.path.join(os.path.dirname(__file__), 'file name')

    print(f"Loading dataset: {excel_filename} ...")
    data = pd.read_excel(excel_filename)
    df_clean = data.copy()
    
    df_clean.columns = [str(c).strip().lower() for c in df_clean.columns]
    id_col = next((c for c in ['playerid', 'player_id', 'id', 'player id'] if c in df_clean.columns), df_clean.columns[0])
    name_col = next((c for c in ['name', 'player_name', 'player name', 'player'] if c in df_clean.columns), None)
    year_col = next((c for c in ['year', 'season', 'yr'] if c in df_clean.columns), None)
    war_col = next((c for c in ['war', 'wins above replacement'] if c in df_clean.columns), None)
    age_col = next((c for c in ['age'] if c in df_clean.columns), None)
    active_col = next((c for c in ['active', 'is_active'] if c in df_clean.columns), None)

    required_mappings = {'id': id_col, 'year': year_col, 'war': war_col, 'age': age_col}
    for key, col in required_mappings.items():
        if col is None or col not in df_clean.columns:
            raise ValueError(f"Critical error: Could not identify mapping for '{key}' in dataset columns!")

    subset_cols = [id_col, year_col, war_col, age_col]
    if active_col:
        subset_cols.append(active_col)
    
    df_clean = df_clean.dropna(subset=subset_cols).copy()
    if year_col:
        df_clean = df_clean[df_clean[year_col] != 2023].copy()
    if name_col:
        df_clean = df_clean[~df_clean[name_col].astype(str).str.contains("Corbin Carroll", case=False, na=False)].copy()
    
    df_clean = df_clean.sort_values(by=[id_col, year_col]).reset_index(drop=True)

    # Feature Engineering
    df_clean['war-1'] = df_clean.groupby(id_col)[war_col].shift(1)
    df_clean['war-2'] = df_clean.groupby(id_col)[war_col].shift(2)
    df_clean['next_season_war'] = df_clean.groupby(id_col)[war_col].shift(-1).fillna(0)

    df_clean['max_player_year'] = df_clean.groupby(id_col)[year_col].transform('max')
    df_clean['is_last_year'] = df_clean[year_col] == df_clean['max_player_year']
    
    if active_col:
        df_clean['next_year'] = np.where(df_clean[active_col] == 1, 1, np.where(df_clean['is_last_year'], 0, 1))
    else:
        df_clean['next_year'] = np.where(df_clean['is_last_year'], 0, 1)

    df_clean = df_clean.drop(columns=['max_player_year', 'is_last_year'])
    D1_base = df_clean.copy()

    rename_dict = {id_col: 'playerid', year_col: 'year', war_col: 'WAR', age_col: 'Age', 'war-1': 'WAR-1', 'war-2': 'WAR-2'}
    if name_col: rename_dict[name_col] = 'Name'
    if active_col: rename_dict[active_col] = 'active'
    else: D1_base['active'] = 1

    D1_base = D1_base.rename(columns=rename_dict)
    D2_base = D1_base.groupby('playerid').agg(
        sum_WAR=('WAR', 'sum'), max_WAR=('WAR', 'max'), max_year=('year', 'max'),
        max_age=('Age', 'max'), Name=('Name', 'first') if 'Name' in D1_base.columns else ('playerid', 'first'),
        active=('active', 'last')
    ).reset_index()

    # =========================================================================
    # CONFIGURABLE REPEAT TIMES (ADJUST HERE: e.g., 1 for single run, 100 for 100 times)
    # =========================================================================
    n_repeats = 100  

    # ==========================================
    # CASE 1: DIRECT MULTI-YEAR WAR REGRESSION
    # ==========================================
    print("\n" + "="*60)
    print(f"EXECUTING CASE 1: DIRECT MULTI-YEAR WAR REGRESSION ({n_repeats} REPEAT(S))")
    print("="*60)

    D1_c1 = D1_base[D1_base['next_year'] != 0].copy().reset_index(drop=True)
    D2_c1 = D2_base.copy().reset_index(drop=True)
    valid_players_c1 = set(D2_c1['playerid'])
    D1_c1 = D1_c1[D1_c1['playerid'].isin(valid_players_c1)].copy().reset_index(drop=True)
    players_c1 = D2_c1['playerid'].values

    for h in range(1, 6):
        D1_c1[f'target_h{h}'] = D1_c1.groupby('playerid')['WAR'].shift(-h)

    c1_models_factory = {
        'Linear Regression': lambda: LinearRegression(),
        'Neural Network (MLP)': lambda: MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42),
        'XGBoost Regressor': lambda: XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
    }

    c1_accumulated = {h: {m: {'cov': [], 'len': [], 'q': [], 'count': 0} for m in c1_models_factory} for h in range(1, 6)}
    fitted_c1_models, fitted_c1_scalers, fitted_c1_quantiles = {}, {}, {}

    for seed in range(n_repeats):
        train_p1, temp_p1 = train_test_split(players_c1, test_size=0.20, random_state=42 + seed)
        cal_p1, test_p1 = train_test_split(temp_p1, test_size=0.50, random_state=42 + seed)

        D2_c1['split'] = 'train'
        D2_c1.loc[D2_c1['playerid'].isin(cal_p1), 'split'] = 'cal'
        D2_c1.loc[D2_c1['playerid'].isin(test_p1), 'split'] = 'test'

        split_map_c1 = dict(zip(D2_c1['playerid'], D2_c1['split']))
        df_run_c1 = D1_c1.copy()
        df_run_c1['split'] = df_run_c1['playerid'].map(split_map_c1)

        for h in range(1, 6):
            target_col = f'target_h{h}'
            df_h = df_run_c1.dropna(subset=[target_col]).copy()

            for model_name, model_fn in c1_models_factory.items():
                df_h[f'pred_{model_name}'] = np.nan

                for y_case in ['y1', 'y2', 'y3plus']:
                    if y_case == 'y1': sub_mask = (df_h['year'] == 1); feats = ['Age', 'year', 'WAR']
                    elif y_case == 'y2': sub_mask = (df_h['year'] == 2); feats = ['Age', 'year', 'WAR', 'WAR-1']
                    else: sub_mask = (df_h['year'] > 2); feats = ['Age', 'year', 'WAR', 'WAR-1', 'WAR-2']

                    train_mask = sub_mask & (df_h['split'] == 'train')
                    if train_mask.sum() == 0: continue

                    scaler = StandardScaler()
                    X_tr = scaler.fit_transform(df_h.loc[train_mask, feats])
                    y_tr = df_h.loc[train_mask, target_col].values

                    mdl = model_fn()
                    mdl.fit(X_tr, y_tr)

                    if seed == 0:
                        fitted_c1_models[(h, model_name, y_case)] = mdl
                        fitted_c1_scalers[(h, model_name, y_case)] = scaler

                    all_mask = sub_mask
                    if all_mask.sum() > 0:
                        X_all = scaler.transform(df_h.loc[all_mask, feats])
                        df_h.loc[all_mask, f'pred_{model_name}'] = mdl.predict(X_all)

                cal_df = df_h[df_h['split'] == 'cal'].copy()
                cal_scores = np.abs(cal_df[target_col] - cal_df[f'pred_{model_name}']).values
                q_global = get_conformal_quantile(cal_scores, alpha=0.10)

                test_df = df_h[df_h['split'] == 'test'].copy()
                test_df['abs_err'] = np.abs(test_df[target_col] - test_df[f'pred_{model_name}'])
                test_df['covered'] = test_df['abs_err'] <= q_global

                cov = test_df['covered'].mean() if len(test_df) > 0 else 0.0
                avg_len = 2.0 * q_global

                c1_accumulated[h][model_name]['cov'].append(cov)
                c1_accumulated[h][model_name]['len'].append(avg_len)
                c1_accumulated[h][model_name]['q'].append(q_global)
                c1_accumulated[h][model_name]['count'] = len(test_df)

                if seed == 0:
                    for y_case in ['y1', 'y2', 'y3plus']:
                        fitted_c1_quantiles[(h, model_name, y_case)] = q_global

    # Print Case 1 Table Format Requested (Averaged over n_repeats)
    print("\nHorizon (H)\tModel Architecture\tTest Row Count\tEmpirical Coverage\tConformal Quantile (q)\tGlobal Avg Interval Length (WAR)")
    for h in range(1, 6):
        for m_name in c1_models_factory:
            acc = c1_accumulated[h][m_name]
            mean_cov = np.mean(acc['cov']) * 100
            mean_q = np.mean(acc['q'])
            mean_len = np.mean(acc['len'])
            print(f"H = {h}\t{m_name}\t{acc['count']:,}\t{mean_cov:.2f}%\t{mean_q:.3f}\t{mean_len:.3f} WAR")


    # ==========================================
    # CASE 2: DIRECT MULTI-YEAR ACTIVE CLASSIFICATION
    # ==========================================
    print("\n" + "="*60)
    print(f"EXECUTING CASE 2: DIRECT MULTI-YEAR ACTIVE CLASSIFICATION ({n_repeats} REPEAT(S))")
    print("="*60)

    D1_c2 = D1_base.copy()
    D2_c2 = D2_base.copy()
    players_c2 = D2_c2['playerid'].values
    season_lookup = set(zip(D1_c2['playerid'], D1_c2['year']))
    max_season = D1_c2['year'].max()

    for h in range(1, 6):
        targets = []
        for idx, row in D1_c2.iterrows():
            future_s = row['year'] + h
            if future_s > max_season:
                targets.append(np.nan)
            else:
                targets.append(1 if (row['playerid'], future_s) in season_lookup else 0)
        D1_c2[f'target_h{h}'] = targets

    c2_models_factory = {
        'Logistic Regression': lambda: LogisticRegression(max_iter=300, random_state=42),
        'MLP Classifier': lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42),
        'XGBoost Classifier': lambda: XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
    }

    c2_accumulated = {h: {m: {'cov': [], 'len': [], 'q': [], 'count': 0} for m in c2_models_factory} for h in range(1, 6)}
    fitted_c2_models, fitted_c2_scalers, fitted_c2_quantiles = {}, {}, {}

    for seed in range(n_repeats):
        train_p2, temp_p2 = train_test_split(players_c2, test_size=0.20, random_state=42 + seed)
        cal_p2, test_p2 = train_test_split(temp_p2, test_size=0.50, random_state=42 + seed)

        D2_c2['split'] = 'train'
        D2_c2.loc[D2_c2['playerid'].isin(cal_p2), 'split'] = 'cal'
        D2_c2.loc[D2_c2['playerid'].isin(test_p2), 'split'] = 'test'

        split_map_c2 = dict(zip(D2_c2['playerid'], D2_c2['split']))
        df_run_c2 = D1_c2.copy()
        df_run_c2['split'] = df_run_c2['playerid'].map(split_map_c2)

        for h in range(1, 6):
            target_col = f'target_h{h}'
            df_h = df_run_c2.dropna(subset=[target_col]).copy()
            df_h[target_col] = df_h[target_col].astype(int)

            for model_name, model_fn in c2_models_factory.items():
                df_h[f'prob0_{model_name}'] = np.nan
                df_h[f'prob1_{model_name}'] = np.nan

                for y_case in ['y1', 'y2', 'y3plus']:
                    if y_case == 'y1': sub_mask = (df_h['year'] == 1); feats = ['Age', 'year', 'WAR']
                    elif y_case == 'y2': sub_mask = (df_h['year'] == 2); feats = ['Age', 'year', 'WAR', 'WAR-1']
                    else: sub_mask = (df_h['year'] > 2); feats = ['Age', 'year', 'WAR', 'WAR-1', 'WAR-2']

                    train_mask = sub_mask & (df_h['split'] == 'train')
                    if train_mask.sum() == 0 or len(df_h.loc[train_mask, target_col].unique()) < 2:
                        continue

                    scaler = StandardScaler()
                    X_tr = scaler.fit_transform(df_h.loc[train_mask, feats])
                    y_tr = df_h.loc[train_mask, target_col].values

                    mdl = model_fn()
                    mdl.fit(X_tr, y_tr)

                    if seed == 0:
                        fitted_c2_models[(h, model_name, y_case)] = mdl
                        fitted_c2_scalers[(h, model_name, y_case)] = scaler

                    all_mask = sub_mask
                    if all_mask.sum() > 0:
                        X_all = scaler.transform(df_h.loc[all_mask, feats])
                        probs = mdl.predict_proba(X_all)
                        if probs.shape[1] == 2:
                            df_h.loc[all_mask, f'prob0_{model_name}'] = probs[:, 0]
                            df_h.loc[all_mask, f'prob1_{model_name}'] = probs[:, 1]
                        else:
                            df_h.loc[all_mask, f'prob0_{model_name}'] = 1.0 - probs[:, 0]
                            df_h.loc[all_mask, f'prob1_{model_name}'] = probs[:, 0]

                cal_df = df_h[df_h['split'] == 'cal'].copy()
                cal_df['score'] = np.where(cal_df[target_col] == 1, 
                                           1.0 - cal_df[f'prob1_{model_name}'], 
                                           1.0 - cal_df[f'prob0_{model_name}'])

                q_global = get_conformal_quantile(cal_df['score'].dropna().values, alpha=0.10)

                test_df = df_h[df_h['split'] == 'test'].copy()
                test_df['inc_0'] = (test_df[f'prob0_{model_name}'] >= (1.0 - q_global))
                test_df['inc_1'] = (test_df[f'prob1_{model_name}'] >= (1.0 - q_global))

                test_df['covered'] = np.where(test_df[target_col] == 1, test_df['inc_1'], test_df['inc_0'])
                test_df['set_length'] = test_df['inc_0'].astype(int) + test_df['inc_1'].astype(int)

                cov = test_df['covered'].mean() if len(test_df) > 0 else 0.0
                avg_len = test_df['set_length'].mean() if len(test_df) > 0 else 0.0

                c2_accumulated[h][model_name]['cov'].append(cov)
                c2_accumulated[h][model_name]['len'].append(avg_len)
                c2_accumulated[h][model_name]['q'].append(q_global)
                c2_accumulated[h][model_name]['count'] = len(test_df)

                if seed == 0:
                    for y_case in ['y1', 'y2', 'y3plus']:
                        fitted_c2_quantiles[(h, model_name, y_case)] = q_global

    # Print Case 2 Table Format Requested (Averaged over n_repeats)
    print("\nHornizol (H)\tModel Architecture\tTest Row Count\tEmpirical Coverage\tConformal Quantile (q)\tGlobal Avg Set Length (Labels)")
    for h in range(1, 6):
        for m_name in c2_models_factory:
            acc = c2_accumulated[h][m_name]
            mean_cov = np.mean(acc['cov']) * 100
            mean_q = np.mean(acc['q'])
            mean_len = np.mean(acc['len'])
            print(f"H = {h}\t{m_name}\t{acc['count']:,}\t{mean_cov:.2f}%\t{mean_q:.3f}\t{mean_len:.3f} labels")


    # ==========================================
    # CORBIN CARROLL SIMULATION (ALGORITHM 3)
    # ==========================================
    def run_corbin_sim(model_type='XGBoost'):
        reg_mname = 'Linear Regression' if model_type == 'Linear/Logistic' else ('Neural Network (MLP)' if model_type == 'Neural Network/MLP' else 'XGBoost Regressor')
        cls_mname = 'Logistic Regression' if model_type == 'Linear/Logistic' else ('MLP Classifier' if model_type == 'Neural Network/MLP' else 'XGBoost Classifier')

        print(f"\n=========================================================================")
        print(f"ALGORITHM 3 (GLOBAL): CORBIN CARROLL CAREER PROJECTION ({model_type.upper()})")
        print(f"=========================================================================")

        d1_hist = [
            {'year': 1, 'Age': 22, 'WAR': 1.4, 'WAR-1': np.nan, 'WAR-2': np.nan},
            {'year': 2, 'Age': 23, 'WAR': 5.4, 'WAR-1': 1.4, 'WAR-2': np.nan}
        ]

        baseline_idx = 1
        active_flag = True
        step_count = 0

        while active_flag and step_count < 25:
            step_count += 1
            base_row = d1_hist[baseline_idx]
            b_year = base_row['year']
            b_age = base_row['Age']

            for h in range(1, 6):
                t_year = b_year + h
                t_age = b_age + h

                if base_row['year'] == 1:
                    y_case = 'y1'
                    feat_vals = [base_row['Age'], base_row['year'], base_row['WAR']]
                elif base_row['year'] == 2:
                    y_case = 'y2'
                    feat_vals = [base_row['Age'], base_row['year'], base_row['WAR'], base_row['WAR-1']]
                else:
                    y_case = 'y3plus'
                    feat_vals = [base_row['Age'], base_row['year'], base_row['WAR'], base_row['WAR-1'], base_row['WAR-2']]

                # Fallback if key missing
                if (h, cls_mname, y_case) not in fitted_c2_scalers:
                    y_case = 'y3plus' if (h, cls_mname, 'y3plus') in fitted_c2_scalers else 'y1'

                scaler_c2 = fitted_c2_scalers[(h, cls_mname, y_case)]
                mdl_c2 = fitted_c2_models[(h, cls_mname, y_case)]
                X_c2 = scaler_c2.transform([feat_vals])
                probs = mdl_c2.predict_proba(X_c2)[0]
                p_active = probs[1] if len(probs) == 2 else probs[0]
                q_c2 = fitted_c2_quantiles[(h, cls_mname, y_case)]

                inc_0 = (probs[0] >= (1.0 - q_c2)) if len(probs) == 2 else True
                inc_1 = (p_active >= (1.0 - q_c2))
                set_len = int(inc_0) + int(inc_1)

                pred_next = 1 if p_active > 0.50 else 0

                if pred_next == 0 and set_len == 1:
                    print(f"\n---> Year {t_year} (Age {t_age}, h={h}): Model 2 CONFIDENTLY predicts RETIRED (P(Active)={p_active:.1%}, Global Set Len=1). Projection Stopped.")
                    active_flag = False
                    break

                scaler_c1 = fitted_c1_scalers[(h, reg_mname, y_case)]
                mdl_c1 = fitted_c1_models[(h, reg_mname, y_case)]
                X_c1 = scaler_c1.transform([feat_vals])
                pred_war = mdl_c1.predict(X_c1)[0]
                q_c1 = fitted_c1_quantiles[(h, reg_mname, y_case)]
                width = 2 * q_c1

                status_str = f"Active {p_active:.1%}" if pred_next == 1 else f"Uncertain {p_active:.1%} (set={{0,1}})"
                print(f"Year {t_year:2d} (Age {t_age:2d}, h={h}): Model2: {status_str} [Set Len={set_len}] | Model1 Pred WAR = {pred_war:+.2f} [90% CI = [{pred_war-q_c1:+.2f}, {pred_war+q_c1:+.2f}], Width = {width:.2f}]")

                d1_hist.append({
                    'year': t_year, 'Age': t_age, 'WAR': pred_war,
                    'WAR-1': d1_hist[-1]['WAR'],
                    'WAR-2': d1_hist[-2]['WAR'] if len(d1_hist) >= 2 else np.nan
                })

            if active_flag:
                baseline_idx = len(d1_hist) - 1
                print(f"\n--- Reached h=5 (Year {d1_hist[baseline_idx]['year']}). Updating baseline to Year {d1_hist[baseline_idx]['year']} for recursive forecasting ---")

    run_corbin_sim('XGBoost')
    run_corbin_sim('Linear/Logistic')
    run_corbin_sim('Neural Network/MLP')


if __name__ == '__main__':
    main()

C:\Users\X413F\anaconda3\lib\site-packages\pandas\core\computation\expressions.py:20: UserWarning: Pandas requires version '2.7.3' or newer of 'numexpr' (version '2.7.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Loading dataset: /Users/X413F/Documents/2023spring/Corbin Carroll Project/2026 revision version/MLB project 2026 version.xlsx ...

EXECUTING CASE 1: DIRECT MULTI-YEAR WAR REGRESSION (1 REPEAT(S))

Horizon (H)	Model Architecture	Test Row Count	Empirical Coverage	Conformal Quantile (q)	Global Avg Interval Length (WAR)
H = 1	Linear Regression	6,682	91.53%	1.961	3.921 WAR
H = 1	Neural Network (MLP)	6,682	90.96%	1.935	3.871 WAR
H = 1	XGBoost Regressor	6,682	91.16%	1.949	3.899 WAR
H = 2	Linear Regression	5,462	91.03%	2.236	4.472 WAR
H = 2	Neural Network (MLP)	5,462	91.07%	2.208	4.416 WAR
H = 2	XGBoost Regressor	5,462	90.59%	2.177	4.354 WAR
H = 3	Linear Regression	4,457	91.36%	2.407	4.814 WAR
H = 3	Neural Network (MLP)	4,457	91.14%	2.428	4.855 WAR
H = 3	XGBoost Regressor	4,457	91.23%	2.401	4.801 WAR
H = 4	Linear Regression	3,628	90.46%	2.492	4.985 WAR
H = 4	Neural Network (MLP)	3,628	90.82%	2.483	4.967 WAR
H = 4	XGBoost Regressor	3,628	90.13%	2.445	4.891 WAR
H = 5	Linear Regression	2,917	91.7

## Locally Adaptive Conformal Prediction

In [2]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neural_network import MLPRegressor, MLPClassifier
from xgboost import XGBRegressor, XGBClassifier

# --- Helper Functions ---
def df_to_md(df, index=False):
    """Converts DataFrame to GitHub Markdown table without external dependencies."""
    if index:
        df = df.reset_index()
    headers = [str(col) for col in df.columns]
    rows = []
    rows.append("| " + " | ".join(headers) + " |")
    rows.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for _, row in df.iterrows():
        str_row = [str(val) for val in row.values]
        rows.append("| " + " | ".join(str_row) + " |")
    return "\n".join(rows)

def get_conformal_quantile(scores, alpha=0.10):
    """Calculates nonconformity score quantile for 1-alpha target coverage."""
    n = len(scores)
    if n == 0:
        return 0.0
    q_val = np.ceil((n + 1) * (1 - alpha)) / n
    q_val = min(1.0, max(0.0, q_val))
    return np.quantile(scores, q_val, method='higher') if hasattr(np, 'quantile') else np.percentile(scores, q_val * 100)

def get_y_case_from_year(year):
    """Map season year to Mondrian bucket."""
    if year == 1:
        return 'y1'
    elif year == 2:
        return 'y2'
    else:
        return 'y3plus'


def main():
    # --- Step 1: Load Data & Preprocess Features ---
    excel_filename = 'file name'
    if not os.path.exists(excel_filename):
        excel_filename = os.path.join(os.path.dirname(__file__), 'file name')

    print(f"Loading dataset: {excel_filename} ...")
    data = pd.read_excel(excel_filename)
    df_clean = data.copy()
    
    df_clean.columns = [str(c).strip().lower() for c in df_clean.columns]
    id_col = next((c for c in ['playerid', 'player_id', 'id', 'player id'] if c in df_clean.columns), df_clean.columns[0])
    name_col = next((c for c in ['name', 'player_name', 'player name', 'player'] if c in df_clean.columns), None)
    year_col = next((c for c in ['year', 'season', 'yr'] if c in df_clean.columns), None)
    war_col = next((c for c in ['war', 'wins above replacement'] if c in df_clean.columns), None)
    age_col = next((c for c in ['age'] if c in df_clean.columns), None)
    active_col = next((c for c in ['active', 'is_active'] if c in df_clean.columns), None)

    required_mappings = {'id': id_col, 'year': year_col, 'war': war_col, 'age': age_col}
    for key, col in required_mappings.items():
        if col is None or col not in df_clean.columns:
            raise ValueError(f"Critical error: Could not identify mapping for '{key}' in dataset columns!")

    subset_cols = [id_col, year_col, war_col, age_col]
    if active_col:
        subset_cols.append(active_col)
    
    df_clean = df_clean.dropna(subset=subset_cols).copy()
    if year_col:
        df_clean = df_clean[df_clean[year_col] != 2023].copy()
    if name_col:
        df_clean = df_clean[~df_clean[name_col].astype(str).str.contains("Corbin Carroll", case=False, na=False)].copy()
    
    df_clean = df_clean.sort_values(by=[id_col, year_col]).reset_index(drop=True)

    # Feature Engineering
    df_clean['war-1'] = df_clean.groupby(id_col)[war_col].shift(1)
    df_clean['war-2'] = df_clean.groupby(id_col)[war_col].shift(2)
    df_clean['next_season_war'] = df_clean.groupby(id_col)[war_col].shift(-1).fillna(0)

    df_clean['max_player_year'] = df_clean.groupby(id_col)[year_col].transform('max')
    df_clean['is_last_year'] = df_clean[year_col] == df_clean['max_player_year']
    
    if active_col:
        df_clean['next_year'] = np.where(df_clean[active_col] == 1, 1, np.where(df_clean['is_last_year'], 0, 1))
    else:
        df_clean['next_year'] = np.where(df_clean['is_last_year'], 0, 1)

    df_clean = df_clean.drop(columns=['max_player_year', 'is_last_year'])
    D1_base = df_clean.copy()

    rename_dict = {id_col: 'playerid', year_col: 'year', war_col: 'WAR', age_col: 'Age', 'war-1': 'WAR-1', 'war-2': 'WAR-2'}
    if name_col: rename_dict[name_col] = 'Name'
    if active_col: rename_dict[active_col] = 'active'
    else: D1_base['active'] = 1

    D1_base = D1_base.rename(columns=rename_dict)
    D2_base = D1_base.groupby('playerid').agg(
        sum_WAR=('WAR', 'sum'), max_WAR=('WAR', 'max'), max_year=('year', 'max'),
        max_age=('Age', 'max'), Name=('Name', 'first') if 'Name' in D1_base.columns else ('playerid', 'first'),
        active=('active', 'last')
    ).reset_index()

    # =========================================================================
    # CONFIGURABLE REPEAT TIMES (ADJUST HERE: e.g., 100 for 100-repeat Monte Carlo)
    # =========================================================================
    n_repeats = 1  

    # ==========================================
    # CASE 1: DIRECT MULTI-YEAR WAR REGRESSION (LOCALLY ADAPTIVE MONDRIAN)
    # ==========================================
    print("\n" + "="*60)
    print(f"EXECUTING CASE 1: DIRECT MULTI-YEAR WAR REGRESSION ({n_repeats} REPEATS - MONDRIAN)")
    print("="*60)

    D1_c1 = D1_base[D1_base['next_year'] != 0].copy().reset_index(drop=True)
    D2_c1 = D2_base.copy().reset_index(drop=True)
    valid_players_c1 = set(D2_c1['playerid'])
    D1_c1 = D1_c1[D1_c1['playerid'].isin(valid_players_c1)].copy().reset_index(drop=True)
    players_c1 = D2_c1['playerid'].values

    for h in range(1, 6):
        D1_c1[f'target_h{h}'] = D1_c1.groupby('playerid')['WAR'].shift(-h)

    c1_models_factory = {
        'Linear Regression': lambda: LinearRegression(),
        'Neural Network (MLP)': lambda: MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42),
        'XGBoost Regressor': lambda: XGBRegressor(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
    }

    c1_accumulated = {h: {m: {'cov': [], 'len': [], 'q': [], 'count': 0} for m in c1_models_factory} for h in range(1, 6)}
    fitted_c1_models, fitted_c1_scalers, fitted_c1_quantiles = {}, {}, {}

    for seed in range(n_repeats):
        train_p1, temp_p1 = train_test_split(players_c1, test_size=0.20, random_state=42 + seed)
        cal_p1, test_p1 = train_test_split(temp_p1, test_size=0.50, random_state=42 + seed)

        D2_c1['split'] = 'train'
        D2_c1.loc[D2_c1['playerid'].isin(cal_p1), 'split'] = 'cal'
        D2_c1.loc[D2_c1['playerid'].isin(test_p1), 'split'] = 'test'

        split_map_c1 = dict(zip(D2_c1['playerid'], D2_c1['split']))
        df_run_c1 = D1_c1.copy()
        df_run_c1['split'] = df_run_c1['playerid'].map(split_map_c1)

        for h in range(1, 6):
            target_col = f'target_h{h}'
            df_h = df_run_c1.dropna(subset=[target_col]).copy()

            for model_name, model_fn in c1_models_factory.items():
                df_h[f'pred_{model_name}'] = np.nan

                for y_case in ['y1', 'y2', 'y3plus']:
                    if y_case == 'y1': sub_mask = (df_h['year'] == 1); feats = ['Age', 'year', 'WAR']
                    elif y_case == 'y2': sub_mask = (df_h['year'] == 2); feats = ['Age', 'year', 'WAR', 'WAR-1']
                    else: sub_mask = (df_h['year'] > 2); feats = ['Age', 'year', 'WAR', 'WAR-1', 'WAR-2']

                    train_mask = sub_mask & (df_h['split'] == 'train')
                    if train_mask.sum() == 0: continue

                    scaler = StandardScaler()
                    X_tr = scaler.fit_transform(df_h.loc[train_mask, feats])
                    y_tr = df_h.loc[train_mask, target_col].values

                    mdl = model_fn()
                    mdl.fit(X_tr, y_tr)

                    if seed == 0:
                        fitted_c1_models[(h, model_name, y_case)] = mdl
                        fitted_c1_scalers[(h, model_name, y_case)] = scaler

                    all_mask = sub_mask
                    if all_mask.sum() > 0:
                        X_all = scaler.transform(df_h.loc[all_mask, feats])
                        df_h.loc[all_mask, f'pred_{model_name}'] = mdl.predict(X_all)

                # Mondrian bucket quantiles
                q_by_ycase = {}
                for y_case in ['y1', 'y2', 'y3plus']:
                    if y_case == 'y1': cal_mask = (df_h['year'] == 1) & (df_h['split'] == 'cal')
                    elif y_case == 'y2': cal_mask = (df_h['year'] == 2) & (df_h['split'] == 'cal')
                    else: cal_mask = (df_h['year'] > 2) & (df_h['split'] == 'cal')

                    cal_df_y = df_h[cal_mask].copy()
                    if len(cal_df_y) == 0:
                        q_y = 0.0
                    else:
                        cal_scores_y = np.abs(cal_df_y[target_col] - cal_df_y[f'pred_{model_name}']).values
                        q_y = get_conformal_quantile(cal_scores_y, alpha=0.10)
                    q_by_ycase[y_case] = q_y
                    if seed == 0:
                        fitted_c1_quantiles[(h, model_name, y_case)] = q_y

                test_df = df_h[df_h['split'] == 'test'].copy()
                if len(test_df) > 0:
                    test_df['q_local'] = test_df['year'].apply(lambda y: q_by_ycase[get_y_case_from_year(y)])
                    test_df['abs_err'] = np.abs(test_df[target_col] - test_df[f'pred_{model_name}'])
                    test_df['covered'] = test_df['abs_err'] <= test_df['q_local']

                    cov = test_df['covered'].mean()
                    avg_len = (2.0 * test_df['q_local']).mean()
                    mean_q = test_df['q_local'].mean()

                    c1_accumulated[h][model_name]['cov'].append(cov)
                    c1_accumulated[h][model_name]['len'].append(avg_len)
                    c1_accumulated[h][model_name]['q'].append(mean_q)
                    c1_accumulated[h][model_name]['count'] = len(test_df)

    # Print Case 1 Table Format
    print("\nHorizon (H)\tModel Architecture\tTest Row Count\tEmpirical Coverage\tConformal Quantile (q)\tGlobal Avg Interval Length (WAR)")
    for h in range(1, 6):
        for m_name in c1_models_factory:
            acc = c1_accumulated[h][m_name]
            mean_cov = np.mean(acc['cov']) * 100
            mean_q = np.mean(acc['q'])
            mean_len = np.mean(acc['len'])
            print(f"H = {h}\t{m_name}\t{acc['count']:,}\t{mean_cov:.2f}%\t{mean_q:.3f}\t{mean_len:.3f} WAR")


    # ==========================================
    # CASE 2: DIRECT MULTI-YEAR ACTIVE CLASSIFICATION (LOCALLY ADAPTIVE MONDRIAN)
    # ==========================================
    print("\n" + "="*60)
    print(f"EXECUTING CASE 2: DIRECT MULTI-YEAR ACTIVE CLASSIFICATION ({n_repeats} REPEATS - MONDRIAN)")
    print("="*60)

    D1_c2 = D1_base.copy()
    D2_c2 = D2_base.copy()
    players_c2 = D2_c2['playerid'].values
    season_lookup = set(zip(D1_c2['playerid'], D1_c2['year']))
    max_season = D1_c2['year'].max()

    for h in range(1, 6):
        targets = []
        for idx, row in D1_c2.iterrows():
            future_s = row['year'] + h
            if future_s > max_season: targets.append(np.nan)
            else: targets.append(1 if (row['playerid'], future_s) in season_lookup else 0)
        D1_c2[f'target_h{h}'] = targets

    c2_models_factory = {
        'Logistic Regression': lambda: LogisticRegression(max_iter=300, random_state=42),
        'MLP Classifier': lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42),
        'XGBoost Classifier': lambda: XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.05, random_state=42)
    }

    c2_accumulated = {h: {m: {'cov': [], 'len': [], 'q': [], 'count': 0} for m in c2_models_factory} for h in range(1, 6)}
    fitted_c2_models, fitted_c2_scalers, fitted_c2_quantiles = {}, {}, {}

    for seed in range(n_repeats):
        train_p2, temp_p2 = train_test_split(players_c2, test_size=0.20, random_state=42 + seed)
        cal_p2, test_p2 = train_test_split(temp_p2, test_size=0.50, random_state=42 + seed)

        D2_c2['split'] = 'train'
        D2_c2.loc[D2_c2['playerid'].isin(cal_p2), 'split'] = 'cal'
        D2_c2.loc[D2_c2['playerid'].isin(test_p2), 'split'] = 'test'

        split_map_c2 = dict(zip(D2_c2['playerid'], D2_c2['split']))
        df_run_c2 = D1_c2.copy()
        df_run_c2['split'] = df_run_c2['playerid'].map(split_map_c2)

        for h in range(1, 6):
            target_col = f'target_h{h}'
            df_h = df_run_c2.dropna(subset=[target_col]).copy()
            df_h[target_col] = df_h[target_col].astype(int)

            for model_name, model_fn in c2_models_factory.items():
                df_h[f'prob0_{model_name}'] = np.nan
                df_h[f'prob1_{model_name}'] = np.nan

                for y_case in ['y1', 'y2', 'y3plus']:
                    if y_case == 'y1': sub_mask = (df_h['year'] == 1); feats = ['Age', 'year', 'WAR']
                    elif y_case == 'y2': sub_mask = (df_h['year'] == 2); feats = ['Age', 'year', 'WAR', 'WAR-1']
                    else: sub_mask = (df_h['year'] > 2); feats = ['Age', 'year', 'WAR', 'WAR-1', 'WAR-2']

                    train_mask = sub_mask & (df_h['split'] == 'train')
                    if train_mask.sum() == 0 or len(df_h.loc[train_mask, target_col].unique()) < 2: continue

                    scaler = StandardScaler()
                    X_tr = scaler.fit_transform(df_h.loc[train_mask, feats])
                    y_tr = df_h.loc[train_mask, target_col].values

                    mdl = model_fn()
                    mdl.fit(X_tr, y_tr)

                    if seed == 0:
                        fitted_c2_models[(h, model_name, y_case)] = mdl
                        fitted_c2_scalers[(h, model_name, y_case)] = scaler

                    all_mask = sub_mask
                    if all_mask.sum() > 0:
                        X_all = scaler.transform(df_h.loc[all_mask, feats])
                        probs = mdl.predict_proba(X_all)
                        if probs.shape[1] == 2:
                            df_h.loc[all_mask, f'prob0_{model_name}'] = probs[:, 0]
                            df_h.loc[all_mask, f'prob1_{model_name}'] = probs[:, 1]
                        else:
                            df_h.loc[all_mask, f'prob0_{model_name}'] = 1.0 - probs[:, 0]
                            df_h.loc[all_mask, f'prob1_{model_name}'] = probs[:, 0]

                # Mondrian bucket quantiles
                q_by_ycase = {}
                for y_case in ['y1', 'y2', 'y3plus']:
                    if y_case == 'y1': cal_mask = (df_h['year'] == 1) & (df_h['split'] == 'cal')
                    elif y_case == 'y2': cal_mask = (df_h['year'] == 2) & (df_h['split'] == 'cal')
                    else: cal_mask = (df_h['year'] > 2) & (df_h['split'] == 'cal')

                    cal_df_y = df_h[cal_mask].copy()
                    if len(cal_df_y) == 0:
                        q_y = 0.0
                    else:
                        cal_df_y['score'] = np.where(
                            cal_df_y[target_col] == 1,
                            1.0 - cal_df_y[f'prob1_{model_name}'],
                            1.0 - cal_df_y[f'prob0_{model_name}']
                        )
                        q_y = get_conformal_quantile(cal_df_y['score'].dropna().values, alpha=0.10)
                    q_by_ycase[y_case] = q_y
                    if seed == 0:
                        fitted_c2_quantiles[(h, model_name, y_case)] = q_y

                test_df = df_h[df_h['split'] == 'test'].copy()
                if len(test_df) > 0:
                    test_df['q_local'] = test_df['year'].apply(lambda y: q_by_ycase[get_y_case_from_year(y)])

                    test_df['inc_0'] = (test_df[f'prob0_{model_name}'] >= (1.0 - test_df['q_local']))
                    test_df['inc_1'] = (test_df[f'prob1_{model_name}'] >= (1.0 - test_df['q_local']))

                    test_df['covered'] = np.where(test_df[target_col] == 1, test_df['inc_1'], test_df['inc_0'])
                    test_df['set_length'] = test_df['inc_0'].astype(int) + test_df['inc_1'].astype(int)

                    cov = test_df['covered'].mean()
                    avg_len = test_df['set_length'].mean()
                    mean_q = test_df['q_local'].mean()

                    c2_accumulated[h][model_name]['cov'].append(cov)
                    c2_accumulated[h][model_name]['len'].append(avg_len)
                    c2_accumulated[h][model_name]['q'].append(mean_q)
                    c2_accumulated[h][model_name]['count'] = len(test_df)

    # Print Case 2 Table Format
    print("\nHornizol (H)\tModel Architecture\tTest Row Count\tEmpirical Coverage\tConformal Quantile (q)\tGlobal Avg Set Length (Labels)")
    for h in range(1, 6):
        for m_name in c2_models_factory:
            acc = c2_accumulated[h][m_name]
            mean_cov = np.mean(acc['cov']) * 100
            mean_q = np.mean(acc['q'])
            mean_len = np.mean(acc['len'])
            print(f"H = {h}\t{m_name}\t{acc['count']:,}\t{mean_cov:.2f}%\t{mean_q:.3f}\t{mean_len:.3f} labels")


    # ==========================================
    # CORBIN CARROLL SIMULATION (ALGORITHM 3)
    # ==========================================
    def run_corbin_sim(model_type='XGBoost'):
        reg_mname = 'Linear Regression' if model_type == 'Linear/Logistic' else ('Neural Network (MLP)' if model_type == 'Neural Network/MLP' else 'XGBoost Regressor')
        cls_mname = 'Logistic Regression' if model_type == 'Linear/Logistic' else ('MLP Classifier' if model_type == 'Neural Network/MLP' else 'XGBoost Classifier')

        print(f"\n=========================================================================")
        print(f"ALGORITHM 3 (LOCAL MONDRIAN): CORBIN CARROLL CAREER PROJECTION ({model_type.upper()})")
        print(f"=========================================================================")

        d1_hist = [
            {'year': 1, 'Age': 22, 'WAR': 1.4, 'WAR-1': np.nan, 'WAR-2': np.nan},
            {'year': 2, 'Age': 23, 'WAR': 5.4, 'WAR-1': 1.4, 'WAR-2': np.nan}
        ]

        baseline_idx = 1
        active_flag = True
        step_count = 0

        while active_flag and step_count < 25:
            step_count += 1
            base_row = d1_hist[baseline_idx]
            b_year = base_row['year']
            b_age = base_row['Age']

            for h in range(1, 6):
                t_year = b_year + h
                t_age = b_age + h

                y_case = get_y_case_from_year(base_row['year'])
                if base_row['year'] == 1:
                    feat_vals = [base_row['Age'], base_row['year'], base_row['WAR']]
                elif base_row['year'] == 2:
                    feat_vals = [base_row['Age'], base_row['year'], base_row['WAR'], base_row['WAR-1']]
                else:
                    feat_vals = [base_row['Age'], base_row['year'], base_row['WAR'], base_row['WAR-1'], base_row['WAR-2']]

                # Fallback if key missing
                if (h, cls_mname, y_case) not in fitted_c2_scalers:
                    y_case = 'y3plus' if (h, cls_mname, 'y3plus') in fitted_c2_scalers else 'y1'

                scaler_c2 = fitted_c2_scalers[(h, cls_mname, y_case)]
                mdl_c2 = fitted_c2_models[(h, cls_mname, y_case)]
                X_c2 = scaler_c2.transform([feat_vals])
                probs = mdl_c2.predict_proba(X_c2)[0]
                p_active = probs[1] if len(probs) == 2 else probs[0]
                q_c2 = fitted_c2_quantiles[(h, model_name, y_case)] if (h, model_name, y_case) in fitted_c2_quantiles else 0.5

                inc_0 = (probs[0] >= (1.0 - q_c2)) if len(probs) == 2 else True
                inc_1 = (p_active >= (1.0 - q_c2))
                set_len = int(inc_0) + int(inc_1)

                pred_next = 1 if p_active > 0.50 else 0

                if pred_next == 0 and set_len == 1:
                    print(f"\n---> Year {t_year} (Age {t_age}, h={h}): Model 2 CONFIDENTLY predicts RETIRED (P(Active)={p_active:.1%}, Local Set Len=1). Projection Stopped.")
                    active_flag = False
                    break

                scaler_c1 = fitted_c1_scalers[(h, reg_mname, y_case)]
                mdl_c1 = fitted_c1_models[(h, reg_mname, y_case)]
                X_c1 = scaler_c1.transform([feat_vals])
                pred_war = mdl_c1.predict(X_c1)[0]
                q_c1 = fitted_c1_quantiles[(h, reg_mname, y_case)]
                width = 2 * q_c1

                status_str = f"Active {p_active:.1%}" if pred_next == 1 else f"Uncertain {p_active:.1%} (set={{0,1}})"
                print(f"Year {t_year:2d} (Age {t_age:2d}, h={h}): Model2: {status_str} [Set Len={set_len}] | Model1 Pred WAR = {pred_war:+.2f} [90% CI = [{pred_war-q_c1:+.2f}, {pred_war+q_c1:+.2f}], Width = {width:.2f}]")

                d1_hist.append({
                    'year': t_year, 'Age': t_age, 'WAR': pred_war,
                    'WAR-1': d1_hist[-1]['WAR'],
                    'WAR-2': d1_hist[-2]['WAR'] if len(d1_hist) >= 2 else np.nan
                })

            if active_flag:
                baseline_idx = len(d1_hist) - 1
                print(f"\n--- Reached h=5 (Year {d1_hist[baseline_idx]['year']}). Updating baseline to Year {d1_hist[baseline_idx]['year']} for recursive forecasting ---")

    run_corbin_sim('XGBoost')
    run_corbin_sim('Linear/Logistic')
    run_corbin_sim('Neural Network/MLP')


if __name__ == '__main__':
    main()

Loading dataset: /Users/X413F/Documents/2023spring/Corbin Carroll Project/2026 revision version/MLB project 2026 version.xlsx ...

EXECUTING CASE 1: DIRECT MULTI-YEAR WAR REGRESSION (1 REPEATS - MONDRIAN)

Horizon (H)	Model Architecture	Test Row Count	Empirical Coverage	Conformal Quantile (q)	Global Avg Interval Length (WAR)
H = 1	Linear Regression	6,682	91.41%	1.911	3.823 WAR
H = 1	Neural Network (MLP)	6,682	91.13%	1.916	3.832 WAR
H = 1	XGBoost Regressor	6,682	91.05%	1.914	3.829 WAR
H = 2	Linear Regression	5,462	90.94%	2.196	4.392 WAR
H = 2	Neural Network (MLP)	5,462	90.79%	2.174	4.347 WAR
H = 2	XGBoost Regressor	5,462	90.57%	2.163	4.325 WAR
H = 3	Linear Regression	4,457	91.56%	2.413	4.827 WAR
H = 3	Neural Network (MLP)	4,457	91.09%	2.426	4.853 WAR
H = 3	XGBoost Regressor	4,457	91.12%	2.396	4.791 WAR
H = 4	Linear Regression	3,628	90.49%	2.486	4.972 WAR
H = 4	Neural Network (MLP)	3,628	90.71%	2.504	5.008 WAR
H = 4	XGBoost Regressor	3,628	89.97%	2.444	4.888 WAR
H = 5	Linear Regression	2